# Combined Orientation + Morphology Map
Links particle segmentation (from `run1_analysis`) with ACOM orientation maps (from `ACOM_analysis`).
For each dataset the figure shows:
- **Left**: ADF image with particle outlines
- **Right**: IPF-Z orientation map, CI-masked, with the same outlines
- **Inset**: IPF colour key

In [1]:
%matplotlib widget
import glob
import os
import re

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from skimage.segmentation import find_boundaries
from skimage import filters, morphology, measure

import hyperspy.api as hs
from orix.io import load as orix_load
from orix.plot import IPFColorKeyTSL, register_projections
from orix.vector import Vector3d

hs.set_log_level('ERROR')
register_projections()

## 1  Paths

In [2]:
base_path  = '/dls/e02/data/2026/cm44133-1/processing/Merlin/Pt_NP_24Feb_Overnight'

ang_files      = sorted(glob.glob(base_path + '/*/ACOM_array/*.ang'))
meta_paths     = sorted(glob.glob(base_path + '/*/*.hdf'))
azim_mean_paths = sorted(glob.glob(base_path + '/*/*_mean.hspy'))

# Extract timestamp from each path so we can align the two lists
def timestamp_from_path(p):
    # Directories are named YYYYMMDD_HHMMSS
    m = re.search(r'(\d{8}_\d{6})', p)
    return m.group(1) if m else None

ang_ts   = [timestamp_from_path(p) for p in ang_files]
meta_ts  = [timestamp_from_path(p) for p in meta_paths]
mean_ts  = [timestamp_from_path(p) for p in azim_mean_paths]

# Build a lookup: timestamp -> index in each list
ang_by_ts  = {ts: p for ts, p in zip(ang_ts,  ang_files)}
meta_by_ts = {ts: p for ts, p in zip(meta_ts, meta_paths)}
mean_by_ts = {ts: p for ts, p in zip(mean_ts, azim_mean_paths)}

# Timestamps that have both ACOM output AND metadata
common_ts = sorted(set(ang_by_ts) & set(meta_by_ts) & set(mean_by_ts))
print(f'{len(common_ts)} datasets with both ang + metadata')

153 datasets with both ang + metadata


In [5]:
# ── Diagnostic: inspect axis ordering of one azimuthal mean file ──────────────
# Run this once. The key question: is the first navigation axis named 'x' or 'y'?
# If it prints  axis 0: name='x'  → the array is (x, y) and will be transposed.
# If it prints  axis 0: name='y'  → already (y, x), no transpose needed.

_d_check = hs.load(azim_mean_paths[0])
print("Azimuthal mean signal:")
_axes_info(_d_check)
del _d_check

Azimuthal mean signal:
  signal type : Signal1D
  data shape  : (255, 255, 365)
  axis 0: name='Navigation'  navigate=True  size=255  scale=5.2083  units='nm'
  axis 1: name='Navigation'  navigate=True  size=255  scale=5.2083  units='nm'
  axis 2: name='Azimuthal Mean'  navigate=False  size=365  scale=0.0056543  units='1/Å'


## 2  Helper: load one dataset

In [4]:
MAP_SHAPE = (255, 255)   # known scan shape
CI_THRESHOLD = 0.1       # pixels below this are treated as background


def _axes_info(sig):
    """Print axes_manager summary — run once to verify axis ordering."""
    am = sig.axes_manager
    print(f"  signal type : {type(sig).__name__}")
    print(f"  data shape  : {sig.data.shape}")
    for ax in am._axes:
        print(f"  axis {ax.index_in_array}: name={ax.name!r:8s}  "
              f"navigate={ax.navigate}  size={ax.size}  "
              f"scale={ax.scale:.5g}  units={ax.units!r}")


def load_adf_image(mean_path, left_q=0.172, right_q=0.229, power=0.1,
                   _debug=False):
    """
    Build an ADF-like image from the azimuthal mean in a q range.

    Returns a 2-D array in (row, col) = (y, x) order so it aligns
    with the (y, x) reshape assumed for the .ang orientation map.

    HyperSpy stores navigation axes slowest-first.  For a 2-D scan
    the convention is axis-0 = y (slow) and axis-1 = x (fast), giving
    .data in (y, x) order — correct for imshow.  If the file was saved
    with x as axis-0 we detect this from the axis *name* and transpose.
    """
    d = hs.load(mean_path)

    if _debug:
        print("Raw signal axes:")
        _axes_info(d)

    # Slice the signal (q) axis by calibrated value and sum.
    # isig slicing keeps navigation axes intact — no .T needed.
    d_roi = d.isig[left_q:right_q]
    adf_sig = d_roi.sum()      # sums signal axis → 2-D navigation image

    if _debug:
        print("After isig slice + sum:")
        _axes_info(adf_sig)

    adf = adf_sig.data  # shape (nav0_size, nav1_size)

    # Guarantee (y, x) order by inspecting navigation axis names.
    nav_axes = [ax for ax in adf_sig.axes_manager._axes if ax.navigate]
    if len(nav_axes) == 2:
        first_name = nav_axes[0].name.lower()
        # If the fastest (first) navigation axis is labelled 'x', transpose.
        if first_name == 'x':
            adf = adf.T
            if _debug:
                print("  → transposed: nav axes were (x, y), corrected to (y, x).")

    return np.power(np.clip(adf, 0, None), power)


def particle_labels_from_adf(adf_image, sigm1=5, sigm2=100,
                              min_size=20, exp_pix=2,
                              ignore_border=True):
    """Segment particles from an ADF image using Difference of Gaussians."""
    from skimage.segmentation import clear_border

    dog = filters.difference_of_gaussians(adf_image, sigm1, sigm2)
    thresh = dog > 0
    cleaned = morphology.remove_small_objects(thresh, min_size=min_size)
    if ignore_border:
        cleaned = clear_border(cleaned)
    expanded = morphology.binary_dilation(cleaned, morphology.disk(exp_pix))
    return measure.label(expanded)


def load_orientation_rgb(ang_path, ci_threshold=CI_THRESHOLD,
                         direction=(0, 0, 1)):
    """Return (rgb_masked, ci_map, sym) for one .ang file."""
    xmap = orix_load(ang_path)

    df = pd.read_csv(ang_path, sep=r'\s+', comment='#',
                     header=None, engine='python')
    df.columns = ['phi1','Phi','phi2','x','y',
                  'image_quality','confidence_index',
                  'phase_id','detector_signal','pattern_fit']

    ci = df['confidence_index'].values.reshape(MAP_SHAPE)
    bg_mask = (ci < ci_threshold)

    sym  = xmap.phases[1].point_group.laue
    ckey = IPFColorKeyTSL(sym, direction=Vector3d(direction))
    O    = xmap['Pt'].orientations
    rgb  = ckey.orientation2color(O).reshape(*MAP_SHAPE, 3)

    rgb_masked = rgb.copy()
    rgb_masked[bg_mask] = 0   # black background

    return rgb_masked, ci, sym

## 3  Single-dataset combined figure

In [6]:
def plot_combined(ts, ci_threshold=CI_THRESHOLD, direction=(0,0,1),
                  outline_color='yellow', outline_lw=0.8,
                  show_particle_ids=True, figsize=(11, 5)):
    """
    Produce a combined ADF + IPF figure for dataset `ts` (timestamp string).
    Returns (fig, axes).
    """
    ang_path  = ang_by_ts[ts]
    mean_path = mean_by_ts[ts]

    # --- load data ---
    adf      = load_adf_image(mean_path)
    labels   = particle_labels_from_adf(adf)
    rgb, ci, sym = load_orientation_rgb(ang_path, ci_threshold, direction)

    # Particle outlines for overlay
    bounds = find_boundaries(labels, mode='outer')

    # --- figure layout ---
    fig = plt.figure(figsize=figsize)
    gs  = fig.add_gridspec(1, 2, wspace=0.05)
    ax0 = fig.add_subplot(gs[0])
    ax1 = fig.add_subplot(gs[1])

    # ---- left panel: ADF + outlines ----
    ax0.imshow(adf, cmap='gray', interpolation='nearest')
    _overlay_boundaries(ax0, bounds, outline_color)
    if show_particle_ids:
        _label_particles(ax0, labels)
    ax0.set_title('ADF image + particle segmentation', fontsize=10)
    ax0.axis('off')

    # ---- right panel: IPF map + outlines ----
    ax1.imshow(rgb, interpolation='nearest')
    _overlay_boundaries(ax1, bounds, outline_color)
    ax1.set_title(f'IPF-Z orientation map  (CI > {ci_threshold})', fontsize=10)
    ax1.axis('off')

    # ---- IPF colour key inset (top-right of right panel) ----
    v = Vector3d(direction)
    ax_key = fig.add_axes([0.73, 0.62, 0.14, 0.30],
                          projection='ipf', symmetry=sym)
    ax_key.plot_ipf_color_key(show_title=False)
    ax_key.patch.set_facecolor('none')
    ax_key.set_title('IPF-Z key', fontsize=7, pad=2)

    fig.suptitle(f'Dataset {ts}', fontsize=11, y=1.01)
    return fig, (ax0, ax1)


def _overlay_boundaries(ax, bounds, color):
    """Draw boundary pixels as a colour overlay."""
    rgba = np.zeros((*bounds.shape, 4), dtype=float)
    c = plt.matplotlib.colors.to_rgba(color)
    rgba[bounds] = c
    ax.imshow(rgba, interpolation='nearest')


def _label_particles(ax, labels):
    """Print the particle ID at each centroid."""
    for prop in measure.regionprops(labels):
        y, x = prop.centroid
        ax.text(x, y, str(prop.label),
                color='white', fontsize=5,
                ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.1', fc='black', alpha=0.4, lw=0))

In [9]:
# Try on one dataset
ts_example = common_ts[0]   # change index to inspect different datasets
fig, axes = plot_combined(ts_example)
plt.tight_layout()
plt.show()

/tmp/ipykernel_354039/3114832951.py:67: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  cleaned = morphology.remove_small_objects(thresh, min_size=min_size)
/tmp/ipykernel_354039/3114832951.py:70: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  expanded = morphology.binary_dilation(cleaned, morphology.disk(exp_pix))


RuntimeError: structure and input must have same dimensionality

## 4  Per-particle orientation summary
For each segmented particle, compute the mean IPF colour and scatter its orientations in IPF space.

In [8]:
def plot_per_particle_ipf(ts, ci_threshold=CI_THRESHOLD,
                          direction=(0, 0, 1), max_particles=12):
    """
    For each particle label, scatter its valid orientations on a shared IPF.
    Each particle gets a unique colour so spread is immediately visible.
    """
    ang_path  = ang_by_ts[ts]
    mean_path = mean_by_ts[ts]

    adf    = load_adf_image(mean_path)
    labels = particle_labels_from_adf(adf)

    xmap = orix_load(ang_path)
    df = pd.read_csv(ang_path, sep=r'\s+', comment='#',
                     header=None, engine='python')
    df.columns = ['phi1','Phi','phi2','x','y',
                  'image_quality','confidence_index',
                  'phase_id','detector_signal','pattern_fit']
    ci = df['confidence_index'].values.reshape(MAP_SHAPE)

    sym  = xmap.phases[1].point_group.laue
    v    = Vector3d(direction)
    O    = xmap['Pt'].orientations.data.reshape(*MAP_SHAPE, 4)  # quaternions

    from orix.quaternion import Rotation

    particle_ids = [p.label for p in measure.regionprops(labels)]
    n = min(len(particle_ids), max_particles)
    cmap = plt.cm.tab20

    fig = plt.figure(figsize=(6, 6))
    ax  = fig.add_subplot(111, projection='ipf', symmetry=sym, direction=v)

    handles = []
    for i, pid in enumerate(particle_ids[:n]):
        mask_p = (labels == pid) & (ci >= ci_threshold)
        if mask_p.sum() == 0:
            continue
        quats = O[mask_p]           # (N, 4)
        O_p   = Rotation(quats)
        O_p.symmetry = sym
        color = cmap(i / max(n - 1, 1))
        ax.scatter(O_p, c=[color] * len(O_p), alpha=0.4, s=8)
        handles.append(mpatches.Patch(color=color, label=f'P{pid}'))

    ax.set_title(f'Per-particle IPF-Z  ({ts})', fontsize=10)
    ax.legend(handles=handles, fontsize=6, loc='lower right',
              ncol=2, framealpha=0.7)

    # colour key inset
    ax_key = fig.add_axes([0.72, 0.72, 0.16, 0.20],
                          projection='ipf', symmetry=sym)
    ax_key.plot_ipf_color_key(show_title=False)
    ax_key.patch.set_facecolor('none')
    return fig, ax


fig2, ax2 = plot_per_particle_ipf(ts_example)
plt.tight_layout()
plt.show()

/tmp/ipykernel_354039/3114832951.py:67: FutureWarning: Parameter `min_size` is deprecated since version 0.26.0 and will be removed in 2.0.0 (or later). To avoid this warning, please use the parameter `max_size` instead. For more details, see the documentation of `remove_small_objects`. Note that the new threshold removes objects smaller than **or equal to** its value, while the previous parameter only removed smaller ones.
  cleaned = morphology.remove_small_objects(thresh, min_size=min_size)
/tmp/ipykernel_354039/3114832951.py:70: FutureWarning: `binary_dilation` is deprecated since version 0.26 and will be removed in version 0.28. Use `skimage.morphology.dilation` instead. Note the lack of mirroring for non-symmetric footprints (see docstring notes).
  expanded = morphology.binary_dilation(cleaned, morphology.disk(exp_pix))


RuntimeError: structure and input must have same dimensionality

## 5  Batch export: save combined figure for every dataset

In [ ]:
out_dir = os.path.join(base_path, 'figures', 'combined_orientation_morphology')
os.makedirs(out_dir, exist_ok=True)

failed = []
for i, ts in enumerate(common_ts):
    try:
        fig, _ = plot_combined(ts, show_particle_ids=False)
        out_path = os.path.join(out_dir, f'{ts}_combined.png')
        fig.savefig(out_path, dpi=150, bbox_inches='tight')
        plt.close(fig)
        print(f'[{i+1}/{len(common_ts)}] saved {ts}')
    except Exception as e:
        print(f'[{i+1}/{len(common_ts)}] FAILED {ts}: {e}')
        failed.append(ts)

print(f'\nDone. {len(failed)} failures:', failed)